# PNAD COVID-19 — Pipeline Raw → Trusted

**Fonte:** IBGE — PNAD COVID-19 (maio–novembro 2020)

Gera as **dimensões** do Star Schema e as **bases temáticas** a partir dos microdados brutos.
Toda renomeação de colunas, cast de tipos e tradução de categorias é dirigida pelos dois dicionários externos.

| Tabela | Tipo | Conteúdo |
|---|---|---|
| `DIM_TEMPO` | Dimensão | Mês, entrevista, fase da pandemia |
| `DIM_PERFIL` | Dimensão | Perfil demográfico com labels traduzidos + faixa etária |
| `DIM_LOCALIZACAO` | Dimensão | UF, região, situação domicílio, capital, RM |
| `DIM_DICIONARIO` | Metadado | Linhagem de todas as 148 variáveis |
| `BASE_SAUDE` | Temática | Sintomas clínicos, comorbidades, testagem |
| `BASE_COMPORTAMENTO` | Temática | Atendimento, máscara, situação laboral |
| `BASE_ECONOMICO` | Temática | Renda, auxílios, benefícios governamentais |
| `BASE_TRABALHO` | Temática | Características detalhadas do trabalho |

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from itertools import chain as ichain

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'raw' / 'dicionario_variaveis.csv').exists()
                     and (p / 'raw' / 'dicionario_categorias.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Não encontrei a raiz do projeto com os dicionários em raw/.')
os.chdir(PROJECT_ROOT)

spark = (SparkSession.builder
    .appName('PNAD_COVID19_raw_to_trusted')
    .config('spark.ui.enabled', 'false')
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.fallback.enabled', 'true')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

# Mesmo critério do ETL-PNAD_COVID-CSV.py (AWS Glue):
# glob com todos os meses (.csv ou .csv.gz — Spark lê gzip nativamente); MES_REF é extraído do nome do arquivo via regexp_extract.
CONFIG = {
    'caminho_microdados':   'raw/PNAD_COVID_*.csv*',
    'caminho_variaveis':    'raw/dicionario_variaveis.csv',
    'caminho_categorias':   'raw/dicionario_categorias.csv',
    'output_base':          'output/pnad_covid',
    'encoding_microdados':  'iso-8859-1',
    'separador_microdados': ',',
}
print('Setup OK | CWD:', os.getcwd())

In [ ]:
dv_pd = pd.read_csv(CONFIG['caminho_variaveis'],  dtype=str, encoding='utf-8')
dc_pd = pd.read_csv(CONFIG['caminho_categorias'], dtype=str, encoding='utf-8')

# Maps a partir de dicionario_variaveis
mapa_renomear = dict(zip(dv_pd['codigo_variavel'], dv_pd['nome_semantico']))
mapa_tipos    = dict(zip(dv_pd['nome_semantico'], dv_pd['tipo']))
mapa_tipos['peso_domicilio_pessoas'] = 'double'  # V1031 vem com casas decimais nos microdados
mapa_dominio  = dict(zip(dv_pd['nome_semantico'], dv_pd['dominio'].fillna('')))
mapa_blocos   = dv_pd.groupby('bloco')['nome_semantico'].apply(list).to_dict()

# Mapa de categorias: {nome_semantico: {int_valor: label}}
# Pula entradas cujo "valor" não seja um código numérico — algumas variáveis contínuas
# (ex.: A002 idade, C008 horas, A001B3 ano_nascimento) têm linhas no dicionário de
# categorias cujo "valor" é uma descrição de domínio ('000 a 130', 'Ano', ...) e não
# um código real. Tratar essas como categoria zeraria a coluna no create_map().
cat_maps = {}
for _, row in dc_pd.iterrows():
    valor = row['valor']
    if pd.isna(valor) or str(valor).strip() == '':
        continue
    try:
        chave = int(float(valor))
    except (ValueError, TypeError):
        continue
    nome = mapa_renomear.get(row['codigo_variavel'], row['codigo_variavel'])
    cat_maps.setdefault(nome, {})[chave] = row['descricao_valor']

CHAVE_JOIN = ['uf', 'id_domicilio', 'id_morador', 'mes_entrevista']
BASE_COLS  = mapa_blocos.get('id', []) + mapa_blocos.get('peso', []) + ['MES_REF']

print(f'Variáveis: {len(dv_pd)} | Categorias: {len(dc_pd)} | '
      f'Colunas com tradução: {len(cat_maps)} | Blocos: {sorted(mapa_blocos)}')

In [ ]:
import glob, re
from functools import reduce

# As colunas dos CSVs estão em POSIÇÕES diferentes entre meses (Nov/2020 tem 3 colunas
# extras inseridas no meio — A006A/A006B/A007A — deslocando todas as seguintes).
# Ler com glob único + header=true desalinha os dados, porque Spark usa o header de
# UM arquivo e mapeia colunas por posição nos outros. Solução: ler cada CSV
# individualmente (cada um com seu próprio header) e unir via unionByName(allowMissingColumns=True).

arquivos_detectados = sorted(glob.glob(CONFIG['caminho_microdados']))
arquivos_por_mes = {}
duplicados_ignorados = []

for caminho in arquivos_detectados:
    nome_arquivo = os.path.basename(caminho)
    m = re.fullmatch(r'PNAD_COVID_(\d{2})(\d{4})\.csv(?:\.gz)?', nome_arquivo)
    if not m:
        continue
    mes_ref = f'{m.group(2)}-{m.group(1)}'
    escolhido = arquivos_por_mes.get(mes_ref)
    if escolhido is None or (escolhido.endswith('.csv.gz') and caminho.endswith('.csv')):
        if escolhido is not None:
            duplicados_ignorados.append(escolhido)
        arquivos_por_mes[mes_ref] = caminho
    else:
        duplicados_ignorados.append(caminho)

arquivos = [arquivos_por_mes[mes_ref] for mes_ref in sorted(arquivos_por_mes)]
if not arquivos:
    raise FileNotFoundError(f"Nenhum microdado encontrado em: {CONFIG['caminho_microdados']}")

print('Arquivos detectados:', [os.path.basename(a) for a in arquivos])
if duplicados_ignorados:
    print('Arquivos duplicados ignorados:', [os.path.basename(a) for a in sorted(duplicados_ignorados)])

dfs = []
for caminho in arquivos:
    m = re.fullmatch(r'PNAD_COVID_(\d{2})(\d{4})\.csv(?:\.gz)?', os.path.basename(caminho))
    if not m:
        continue
    mes_ref = f'{m.group(2)}-{m.group(1)}'   # MMAAAA → AAAA-MM
    df_m = (spark.read
        .option('header', 'true')
        .option('sep', CONFIG['separador_microdados'])
        .option('encoding', CONFIG['encoding_microdados'])
        .option('inferSchema', 'false')
        .csv(caminho)
        .withColumn('MES_REF', F.lit(mes_ref)))
    dfs.append(df_m)
    print(f'  {os.path.basename(caminho):<24} MES_REF={mes_ref} | colunas={len(df_m.columns)}')

df_raw = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

codigos_validos = set(df_raw.columns) & set(mapa_renomear)
ausentes = set(mapa_renomear) - set(df_raw.columns)
if ausentes:
    print(f'Variáveis ausentes no microdado consolidado: {len(ausentes)} — {sorted(ausentes)}')

df_renamed = df_raw.select(
    [F.col(c).alias(mapa_renomear[c]) for c in codigos_validos] + [F.col('MES_REF')]
)

def aplicar_tipos(df, mapa):
    for col, tipo in mapa.items():
        if col not in df.columns:
            continue
        if tipo == 'integer':
            df = df.withColumn(col, F.col(col).cast(IntegerType()))
        elif tipo == 'double':
            df = df.withColumn(col, F.col(col).cast(DoubleType()))
    return df

df_typed = aplicar_tipos(df_renamed, mapa_tipos)

def traduzir_categorias(df, cat_maps, mapa_dominio, mapa_tipos):
    """Substitui valores inteiros pelas descrições do dicionario_categorias."""
    for nome, mp in cat_maps.items():
        if nome not in df.columns:
            continue
        if mapa_tipos.get(nome) == 'double':      # variável contínua por tipo — pula
            continue
        if mapa_dominio.get(nome, '').strip():    # variável contínua por domínio — pula
            continue
        flat = [F.lit(x) for x in ichain(*mp.items())]
        if not flat:
            continue
        mapa_expr = F.create_map(flat)
        df = df.withColumn(nome, mapa_expr[F.col(nome)])
    return df

df_labeled = traduzir_categorias(df_typed, cat_maps, mapa_dominio, mapa_tipos)
df_labeled = df_labeled.cache()

def montar_base(bloco, extra_cols=None):
    cols = mapa_blocos.get(bloco, [])
    all_cols = BASE_COLS + cols + (extra_cols or [])
    disponiveis = [c for c in all_cols if c in df_labeled.columns]
    seen = set()
    return df_labeled.select([c for c in disponiveis if not (c in seen or seen.add(c))])

print(f'Microdado: {df_labeled.count():,} linhas | {len(df_labeled.columns)} colunas')
print('Meses detectados:',
      sorted([r['MES_REF'] for r in df_labeled.select('MES_REF').distinct().collect()]))

## Dimensões

In [ ]:
dim_tempo = (df_labeled
    .select('mes_entrevista', 'num_entrevista', 'MES_REF')
    .distinct()
    .withColumn('fase_pandemia',
        F.when(F.col('mes_entrevista').isin(5, 6),    'Inicio da Pandemia')
         .when(F.col('mes_entrevista').isin(7, 8, 9), 'Pico da Primeira Onda')
         .otherwise('Desaceleracao')))

In [ ]:
# Todas as vars do bloco perfil já têm labels traduzidos via dicionario_categorias.
# Só faixa_etaria precisa ser derivada, pois idade é variável contínua.
perfil_cols = [c for c in mapa_blocos.get('perfil', []) if c in df_labeled.columns]

dim_perfil = (df_labeled
    .select(CHAVE_JOIN + ['MES_REF'] + perfil_cols)
    .withColumn('faixa_etaria',
        F.when(F.col('idade') < 18,  '00-17')
         .when(F.col('idade') < 30,  '18-29')
         .when(F.col('idade') < 45,  '30-44')
         .when(F.col('idade') < 60,  '45-59')
         .when(F.col('idade') < 75,  '60-74')
         .otherwise('75+')))

In [ ]:
# situacao_domicilio (V1022) = 'Urbana'/'Rural' após tradução do dicionário.
# mora_na_capital (CAPITAL) = nome da capital estadual ou null.
# mora_em_regiao_metropolitana (RM_RIDE) = nome da RM ou null.
# uf já foi traduzido para nome do estado — regiao é derivada via isin com nomes.
loc_cols = [c for c in
    ['situacao_domicilio', 'mora_na_capital', 'mora_em_regiao_metropolitana']
    if c in df_labeled.columns]

norte       = ['Rondônia','Acre','Amazonas','Roraima','Pará','Amapá','Tocantins']
nordeste    = ['Maranhão','Piauí','Ceará','Rio Grande do Norte','Paraíba',
               'Pernambuco','Alagoas','Sergipe','Bahia']
sudeste     = ['Minas Gerais','Espírito Santo','Rio de Janeiro','São Paulo']
sul         = ['Paraná','Santa Catarina','Rio Grande do Sul']
centro_oeste = ['Mato Grosso do Sul','Mato Grosso','Goiás','Distrito Federal']

dim_localizacao = (df_labeled
    .select(CHAVE_JOIN + ['MES_REF'] + loc_cols)
    .withColumn('regiao',
        F.when(F.col('uf').isin(norte),       'Norte')
         .when(F.col('uf').isin(nordeste),    'Nordeste')
         .when(F.col('uf').isin(sudeste),     'Sudeste')
         .when(F.col('uf').isin(sul),         'Sul')
         .when(F.col('uf').isin(centro_oeste),'Centro-Oeste')
         .otherwise(None)))

In [ ]:
dim_dicionario = (spark.read
    .option('header', 'true')
    .option('encoding', 'utf-8')
    .csv(CONFIG['caminho_variaveis'])
    .select('codigo_variavel', 'nome_semantico', 'descricao_variavel',
            'bloco', 'tipo', 'dominio', 'parte', 'meses')
    .withColumn('tipo',
        F.when(F.col('nome_semantico') == 'peso_domicilio_pessoas', F.lit('double'))
         .otherwise(F.col('tipo')))
)

## Bases Temáticas

In [ ]:
base_saude = montar_base('saude')

sintomas = [c for c in [
    'sintoma_febre', 'sintoma_tosse', 'sintoma_dificuldade_respirar',
    'sintoma_fadiga', 'sintoma_perda_olfato_paladar',
] if c in base_saude.columns]

# Após tradução, os valores são 'Sim'/'Não'/'Não sabe'/'Ignorado'
cond_sintoma = F.lit(False)
for s in sintomas:
    cond_sintoma = cond_sintoma | (F.col(s) == 'Sim')

qtd_expr = (sum(F.when(F.col(s) == 'Sim', 1).otherwise(0) for s in sintomas)
            if sintomas else F.lit(0))

base_saude = (base_saude
    .withColumn('ind_teve_sintoma_gripal', F.when(cond_sintoma, 1).otherwise(0))
    .withColumn('qtd_sintomas_relatados',  qtd_expr))

In [ ]:
base_comportamento = montar_base('comportamento')

# Após tradução: trabalhou_na_semana = 'Sim'/'Não',
# estava_afastado_com_vinculo = 'Sim'/'Não', procurou_trabalho_semana = 'Sim'/'Não'
base_comportamento = base_comportamento.withColumn(
    'situacao_mercado_trabalho',
    F.when(F.col('trabalhou_na_semana') == 'Sim',
               'Ocupado - trabalhou')
     .when((F.col('trabalhou_na_semana') == 'Não') &
           (F.col('estava_afastado_com_vinculo') == 'Sim'),
               'Ocupado - afastado')
     .when((F.col('trabalhou_na_semana') == 'Não') &
           (F.col('estava_afastado_com_vinculo') == 'Não') &
           (F.col('procurou_trabalho_semana') == 'Sim'),
               'Desocupado')
     .when((F.col('trabalhou_na_semana') == 'Não') &
           (F.col('estava_afastado_com_vinculo') == 'Não') &
           (F.col('procurou_trabalho_semana') == 'Não'),
               'Fora da forca de trabalho')
     .otherwise('Nao se aplica'))

In [ ]:
base_economico = montar_base('economico')

# renda_efetiva_trabalho_principal (C011A12) = valor efetivamente recebido na semana (double, R$)
# renda_habitual_trabalho_principal (C011A)   = indicador de participação (sempre 1)
# O valor habitual real está em valor_dinheiro (C01012) no bloco trabalho.
# Indicador derivado: se recebeu alguma renda efetiva na semana
if 'renda_efetiva_trabalho_principal' in base_economico.columns:
    base_economico = base_economico.withColumn(
        'ind_tem_renda_efetiva',
        F.when(F.col('renda_efetiva_trabalho_principal') > 0, 1).otherwise(0)
    )

In [ ]:
base_trabalho = montar_base('trabalho')

## Persistência e Validação

In [ ]:
OUTPUT = CONFIG['output_base']
import shutil
import pyarrow.parquet as pq

def resolver_saida(path):
    output_path = Path(path)
    if not output_path.is_absolute():
        output_path = PROJECT_ROOT / output_path
    return output_path.resolve()

def limpar_saida(path):
    output_path = resolver_saida(path)
    allowed_root = (PROJECT_ROOT / OUTPUT).resolve()
    if allowed_root != output_path and allowed_root not in output_path.parents:
        raise ValueError(f'Caminho fora da pasta de saida permitida: {output_path}')
    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    return output_path

def salvar_parquet_portatil(nome, df, path):
    # Evita depender do writer local do Hadoop/winutils. O layout gerado continua
    # compativel com Spark: pastas MES_REF=AAAA-MM para tabelas particionadas.
    output_path = limpar_saida(path)

    if nome == 'DIM_DICIONARIO':
        pdf = df.toPandas()
        pdf.to_parquet(
            output_path / 'part-00000.snappy.parquet',
            index=False,
            engine='pyarrow',
            compression='snappy'
        )
        return

    meses = [r['MES_REF'] for r in df.select('MES_REF').distinct().orderBy('MES_REF').collect()]
    for mes_ref in meses:
        part_path = output_path / f'MES_REF={mes_ref}'
        part_path.mkdir(parents=True, exist_ok=True)

        pdf = df.filter(F.col('MES_REF') == mes_ref).drop('MES_REF').toPandas()
        pdf.to_parquet(
            part_path / f'part-00000-{nome.lower()}.snappy.parquet',
            index=False,
            engine='pyarrow',
            compression='snappy'
        )

def ler_parquet_portatil(path):
    dataset_path = resolver_saida(path)
    parquet_files = sorted(str(p.resolve()) for p in dataset_path.rglob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'Nenhum arquivo parquet encontrado em: {dataset_path}')
    return spark.read.option('basePath', str(dataset_path)).parquet(*parquet_files)

def contar_linhas_parquet_portatil(path):
    dataset_path = resolver_saida(path)
    parquet_files = sorted(dataset_path.rglob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'Nenhum arquivo parquet encontrado em: {dataset_path}')
    return sum(pq.ParquetFile(p).metadata.num_rows for p in parquet_files)

tabelas = [
    ('DIM_TEMPO',          dim_tempo),
    ('DIM_PERFIL',         dim_perfil),
    ('DIM_LOCALIZACAO',    dim_localizacao),
    ('DIM_DICIONARIO',     dim_dicionario),
    ('BASE_SAUDE',         base_saude),
    ('BASE_COMPORTAMENTO', base_comportamento),
    ('BASE_ECONOMICO',     base_economico),
    ('BASE_TRABALHO',      base_trabalho),
]

for nome, df in tabelas:
    path = f'{OUTPUT}/{nome.lower()}'
    print(f'Gravando {nome:<25} -> {path}')
    salvar_parquet_portatil(nome, df, path)

print('\n=== Validacao ===')
for nome, _ in tabelas:
    path = f'{OUTPUT}/{nome.lower()}'
    n = contar_linhas_parquet_portatil(path)
    print(f'{nome:<25} {n:>10,} linhas')


In [ ]:
spark.stop()
print('Pipeline finalizado.')